# Model Evaluation and Visualization System


## Environment Initialization


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import torchvision.models as models
from tqdm import tqdm
import open_clip
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = ['DejaVu Sans']

print("Environment setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
 print(f"GPU: {torch.cuda.get_device_name()}")


## Data Path Configuration


In [ ]:
# ==================== Path Configuration ====================
import os
import sys

# Import common constants
sys.path.insert(0, os.getcwd())
from src.config.constants import CLASS_NAMES, NUM_CLASSES

# ==================== Path Configuration ====================

# Data paths - try both possible locations
possible_data_roots = [
 "/root/autodl-tmp/classification/AI4BrainTumor/data",
 "data",
 "data"
]

DATA_ROOT = None
for path in possible_data_roots:
 test_path = os.path.join(path, "test")
 if os.path.exists(test_path):
 DATA_ROOT = path
 print(f" Found data at: {DATA_ROOT}")
 break

if DATA_ROOT is None:
 print(f" Test data not found in any of the expected locations")
 print(f" Please modify DATA_ROOT in this cell")
 DATA_ROOT = "/root/autodl-tmp/classification/AI4BrainTumor/data" # default

TEST_DIR = os.path.join(DATA_ROOT, "test")

# Model checkpoint paths - using parent directory
BEST_MODELS_DIR = "results/best_models"

# Results save path - save to current directory
RESULTS_DIR = "results"

# Verify paths
print("\n" + "="*70)
print("PATH CONFIGURATION")
print("="*70)
print(f"Test set: {TEST_DIR}")
print(f" Exists: {os.path.exists(TEST_DIR)}")
if os.path.exists(TEST_DIR):
 from pathlib import Path
 test_files = list(Path(TEST_DIR).rglob('*.jpg')) + list(Path(TEST_DIR).rglob('*.png'))
 print(f" Files: {len(test_files)}")

print(f"\nModel checkpoints: {BEST_MODELS_DIR}")
print(f" Exists: {os.path.exists(BEST_MODELS_DIR)}")
if os.path.exists(BEST_MODELS_DIR):
 model_files = [f for f in os.listdir(BEST_MODELS_DIR) if f.endswith('.pth')]
 print(f" Models: {len(model_files)}")
 for mf in model_files[:5]: # Show first 5
 print(f" - {mf}")
 if len(model_files) > 5:
 print(f" ... and {len(model_files)-5} more")

print(f"\nClasses: {NUM_CLASSES}")
print(f"Class names: {CLASS_NAMES}")
print("="*70)


## Evaluate Single-Modal DenseNet (Baseline Model)


In [ ]:
# ==================== Import Common Modules ====================

import time
from datetime import datetime
import os
import json

# Import DenseNet classifier (replaces previous FixedDenseNetBaseline)
from src.models.densenet_variants import DenseNetClassifier

# Create alias for backward compatibility
FixedDenseNetBaseline = DenseNetClassifier

print("DenseNet model imported (from common module)")


## Load Test Dataset


In [ ]:
# ==================== Load Test Dataset ====================

# Use paths from Cell 4 configuration
# DATA_ROOT and TEST_DIR already defined above

# Test set transform (same as training val/test)
test_transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load test dataset
if os.path.exists(TEST_DIR):
 test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)
 test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
 print(f" Test set loaded: {len(test_dataset)} samples")
 print(f" Classes: {test_dataset.classes}")
else:
 print(f" Test directory not found: {TEST_DIR}")
 test_loader = None

## Evaluate Single-Modal Models


In [ ]:
# ==================== Evaluate Single-Modal Models ====================

# Import ModelFactory for dynamic model creation
from src.models.single_modal.model_factory import ModelFactory

def evaluate_single_modal_model(model_path, test_loader, device='cuda'):
 """Evaluate single modal model on test set"""
 print(f"\n Evaluating: {os.path.basename(model_path)}")
 
 # Load checkpoint first to get model info
 try:
 checkpoint = torch.load(model_path, map_location=device)
 
 # Extract model information from checkpoint
 model_name = checkpoint.get('model_name', 'Unknown')
 class_names = checkpoint.get('class_names', CLASS_NAMES)
 optimizer_name = checkpoint.get('optimizer_name', 'Unknown')
 lr = checkpoint.get('lr', 0)
 val_acc = checkpoint.get('val_acc', 0)
 best_epoch = checkpoint.get('epoch', -1)
 
 print(f" Model: {model_name}")
 print(f" Optimizer: {optimizer_name}, LR: {lr}")
 print(f" Validation accuracy (training): {val_acc:.4f}")
 print(f" Best epoch: {best_epoch + 1}")
 
 except Exception as e:
 print(f" Failed to load checkpoint: {e}")
 return None
 
 # Create model using ModelFactory
 try:
 factory = ModelFactory()
 model = factory.create_model(model_name, num_classes=len(class_names))
 model = model.to(device)
 
 # Load model weights
 model.load_state_dict(checkpoint['model_state_dict'])
 model.eval()
 
 # Count model parameters
 total_params = sum(p.numel() for p in model.parameters())
 trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
 
 print(f" Model loaded successfully")
 print(f" Parameters: {total_params:,} total, {trainable_params:,} trainable")
 
 except Exception as e:
 print(f" Failed to create/load model: {e}")
 return None
 
 # Evaluate on test set
 model.eval()
 correct = 0
 total = 0
 all_preds = []
 all_labels = []
 all_probs = [] # Collect prediction probabilities for ROC curves
 
 with torch.no_grad():
 for images, labels in tqdm(test_loader, desc=" Testing"):
 images, labels = images.to(device), labels.to(device)
 outputs = model(images)
 
 # Get prediction probabilities
 probs = torch.nn.functional.softmax(outputs, dim=1)
 
 _, predicted = torch.max(outputs, 1)
 
 total += labels.size(0)
 correct += (predicted == labels).sum().item()
 
 all_preds.extend(predicted.cpu().numpy())
 all_labels.extend(labels.cpu().numpy())
 all_probs.extend(probs.cpu().numpy())
 
 test_acc = correct / total
 print(f" Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
 
 # Compare validation and test accuracy
 acc_diff = test_acc - val_acc
 print(f" Val vs Test: {val_acc:.4f} → {test_acc:.4f} ({acc_diff:+.4f})")
 
 # Calculate confusion matrix and classification report
 from sklearn.metrics import confusion_matrix, classification_report
 cm = confusion_matrix(all_labels, all_preds)
 report_dict = classification_report(all_labels, all_preds, 
 target_names=class_names,
 output_dict=True, zero_division=0)
 
 return {
 'model_name': model_name,
 'file_name': os.path.basename(model_path),
 'test_accuracy': test_acc,
 'val_accuracy': val_acc,
 'best_epoch': best_epoch + 1,
 'optimizer': optimizer_name,
 'lr': lr,
 'predictions': all_preds,
 'labels': all_labels,
 'probabilities': all_probs, # for ROC curves
 'confusion_matrix': cm.tolist(), # for visualization
 'results/evaluation_reports/classification_report': report_dict, # for visualization
 'param_count': total_params, # modelparameterscount（for bubble chart）
 'trainable_params': trainable_params,
 'total_samples': total
 }

# Find and evaluate all models
single_modal_results = {}
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.path.exists(BEST_MODELS_DIR) and test_loader is not None:
 model_files = [f for f in os.listdir(BEST_MODELS_DIR) if f.endswith('.pth')]
 print(f"\n Found {len(model_files)} model checkpoints")
 
 for model_file in sorted(model_files):
 model_path = os.path.join(BEST_MODELS_DIR, model_file)
 result = evaluate_single_modal_model(model_path, test_loader, device)
 if result:
 single_modal_results[model_file] = result
 
 # Display summary
 print(f"\n" + "="*70)
 print(f"Single-Modal Models Test Results Summary")
 print("="*70)
 sorted_results = sorted(single_modal_results.items(), 
 key=lambda x: x[1]['test_accuracy'], 
 reverse=True)
 
 for i, (file_name, result) in enumerate(sorted_results, 1):
 model_name = result.get('model_name', file_name)
 test_acc = result['test_accuracy']
 val_acc = result.get('val_accuracy', 0)
 optimizer = result.get('optimizer', 'N/A')
 lr = result.get('lr', 0)
 
 print(f"\n[{i}] {model_name}")
 print(f" Config: {optimizer}, LR={lr}")
 print(f" Val Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
 print(f" Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
 print(f" Difference: {(test_acc-val_acc):+.4f}")
 
 print("="*70)
else:
 print(f" Cannot evaluate models. Check paths.")

## Evaluate CLIP Fusion Model


In [ ]:
# ==================== Evaluate CLIP Fusion Model (Optional) ====================

# Check if fusion model results exist
fusion_results_pattern = "fusion_model_results_*.json"
import glob

fusion_results_files = glob.glob(fusion_results_pattern)
fusion_model_result = None

if fusion_results_files:
 # Load the latest fusion-model results
 latest_fusion_file = sorted(fusion_results_files)[-1]
 print(f" Found CLIP fusion model results: {latest_fusion_file}")
 
 with open(latest_fusion_file, 'r') as f:
 fusion_data = json.load(f)
 
 # ⭐ Extract best test-set accuracy (highlight CLIP)
 # Priority: best_test_accuracy_ever > max(training_history.test_acc) > final_test_accuracy
 best_test_ever = fusion_data.get('best_test_accuracy_ever')
 best_test_epoch = fusion_data.get('best_test_epoch')
 
 # If best_test_accuracy_ever is not available, extract it from training_history
 if best_test_ever is None and 'training_history' in fusion_data:
 history = fusion_data['training_history']
 if 'test_acc' in history and history['test_acc']:
 best_test_ever = max(history['test_acc'])
 # Find the corresponding epoch (now recorded every epoch)
 max_idx = history['test_acc'].index(best_test_ever)
 # Get epoch directly from the epochs array, or compute it (max_idx corresponds to epoch max_idx+1)
 # Determine whether this is the old format (every 5 epochs) or the new format (every epoch)
 test_acc_len = len(history['test_acc'])
 epochs_len = len(history.get('epochs', []))
 
 if test_acc_len == epochs_len:
 # New format: recorded every epoch, direct correspondence
 best_test_epoch = history['epochs'][max_idx] if 'epochs' in history else max_idx + 1
 else:
 # Old format: recorded every 5 epochs
 best_test_epoch = (max_idx + 1) * 5
 print(f" Extracted from history: best test = {best_test_ever:.4f} ({best_test_ever*100:.2f}%) at epoch {best_test_epoch}")
 
 # Final fallback
 if best_test_ever is None:
 best_test_ever = fusion_data.get('final_test_accuracy', 0)
 
 fusion_model_result = {
 'name': fusion_data.get('name', 'CLIP Fusion Model'),
 'model_name': 'CLIP_Fusion_Model',
 'test_accuracy': best_test_ever, # ⭐ Use best test-set accuracy
 'best_test_accuracy_ever': best_test_ever,
 'best_test_epoch': best_test_epoch,
 'best_val_epoch': fusion_data.get('best_val_epoch', fusion_data.get('best_epoch', 'N/A')),
 'test_at_best_val': fusion_data.get('test_at_best_val', fusion_data.get('final_test_accuracy', 0)),
 'total_epochs': fusion_data.get('total_epochs', 'N/A'),
 'clip_available': fusion_data.get('clip_available', False),
 # Addevaluationdata(if available)
 'predictions': fusion_data.get('predictions', []),
 'labels': fusion_data.get('true_labels', []),
 'test_labels': fusion_data.get('test_labels', []),
 'test_probabilities': fusion_data.get('test_probabilities', []),
 'confusion_matrix': fusion_data.get('confusion_matrix', []),
 'test_report': fusion_data.get('test_report', {}),
 'final_test_acc': best_test_ever # ⭐ Usebesttestaccuracy
 }
 
 print(f" ⭐ Best test accuracy: {fusion_model_result['best_test_accuracy_ever']:.4f} ({fusion_model_result['best_test_accuracy_ever']*100:.2f}%)")
 print(f" ⭐ Best test epoch: {fusion_model_result['best_test_epoch']}/{fusion_model_result['total_epochs']}")
 print(f" Test at best val: {fusion_model_result['test_at_best_val']:.4f}")
 print(f" Best val epoch: {fusion_model_result['best_val_epoch']}")
 print(f" CLIP available: {fusion_model_result['clip_available']}")
 
 # Check whether full evaluation data is available
 if fusion_model_result['predictions'] and fusion_model_result['confusion_matrix']:
 print(f" Complete evaluation data available (for confusion matrix)")
else:
 print("ℹ No CLIP fusion model results found")
 print(" This is optional - you can still visualize single-modal results")

## Generate Comprehensive Visualization Charts


In [ ]:
# ==================== Import Complete Visualization System ====================

# Import complete visualization system from models_comparation.ipynb
from src.visualization.complete_visualization import generate_all_visualizations

print(" Complete visualization system imported")

# ==================== Generate All Visualizations ====================

if single_modal_results:
 print(f"\n Preparing data for visualization...")
 
 # willconvert results toformat expected by visualization functions
 # Original function expects: test_results is a list containing dict
 
 viz_single_modal = []
 for file_name, result in single_modal_results.items():
 # Build the complete format expected by the visualization functions
 test_result_data = {
 'true_labels': result['labels'],
 'predictions': result['predictions'],
 'test_accuracy': result['test_accuracy'],
 'accuracy': result['test_accuracy'] # Some functions use this field
 }
 
 # Add probabilities (for ROC curves)
 if 'probabilities' in result and result['probabilities']:
 test_result_data['probabilities'] = result['probabilities']
 
 # Add confusion matrix and classification report
 if 'confusion_matrix' in result:
 test_result_data['confusion_matrix'] = result['confusion_matrix']
 if 'results/evaluation_reports/classification_report' in result:
 test_result_data['test_report'] = result['results/evaluation_reports/classification_report']
 
 viz_result = {
 'name': result.get('model_name', file_name.replace('.pth', '')),
 'test_accuracy': result['test_accuracy'],
 'param_count': result.get('param_count', 0), # modelparameterscount（for bubble chart）
 'model_complexity': 1.0, # modelcomplexity(default 1.0)
 'test_results': [test_result_data] # list format
 }
 
 viz_single_modal.append(viz_result)
 
 print(f" Prepared {len(viz_single_modal)} single-modal results")
 
 # PrepareCLIPFusion result (if available)
 viz_clip = None
 if fusion_model_result:
 # Check whether full evaluation data is available
 if fusion_model_result.get('predictions') and fusion_model_result.get('confusion_matrix'):
 # Full data available; includes confusion matrix
 viz_clip = [{
 'name': fusion_model_result.get('name', 'CLIP Fusion Model'),
 'test_accuracy': fusion_model_result['test_accuracy'],
 'final_test_acc': fusion_model_result.get('final_test_acc', fusion_model_result['test_accuracy']),
 'confusion_matrix': fusion_model_result['confusion_matrix'],
 'test_report': fusion_model_result.get('test_report', {}),
 'test_labels': fusion_model_result.get('test_labels', []),
 'test_probabilities': fusion_model_result.get('test_probabilities', []),
 'best_epoch': fusion_model_result.get('best_epoch', 'N/A')
 }]
 print(f" CLIP fusion result prepared (with confusion matrix)")
 else:
 # Basic info only; no confusion matrix
 viz_clip = [{
 'name': 'CLIP Fusion Model',
 'test_accuracy': fusion_model_result['test_accuracy'],
 'final_test_acc': fusion_model_result.get('final_test_acc', fusion_model_result['test_accuracy']),
 'best_epoch': fusion_model_result.get('best_epoch', 'N/A')
 }]
 print(f" ℹ CLIP fusion result prepared (basic info only, no confusion matrix)")
 
 # Generate all visualization charts
 print(f"\n")
 viz_paths = generate_all_visualizations(
 single_modal_results=viz_single_modal,
 clip_results=viz_clip,
 save_dir="results/visualizations"
 )
 
 if viz_paths:
 print(f"\n All visualizations successfully generated!")
 print(f" Check results/visualizations/ directory")
 else:
 print(f"\n Visualization generation failed")
else:
 print(" No results available for visualization")

## Generate Detailed Report


In [ ]:
# ==================== Generate Final Report ====================

def generate_evaluation_report(results_dict, fusion_result=None):
 """Generate comprehensive evaluation report"""
 
 print(f"\n" + "="*70)
 print(f" COMPREHENSIVE EVALUATION REPORT")
 print("="*70)
 
 # Single-modal models summary
 print(f"\n Single-Modal Models:")
 sorted_results = sorted(results_dict.items(), 
 key=lambda x: x[1]['test_accuracy'], 
 reverse=True)
 
 for i, (name, result) in enumerate(sorted_results, 1):
 print(f" [{i}] {name}")
 print(f" Test Accuracy: {result['test_accuracy']:.4f} ({result['test_accuracy']*100:.2f}%)")
 print(f" Samples: {result['total_samples']}")
 
 # Best single-modal model
 best_name, best_result = sorted_results[0]
 print(f"\n Best Single-Modal Model: {best_name}")
 print(f" Accuracy: {best_result['test_accuracy']:.4f}")
 
 # Fusion model comparison
 if fusion_result:
 print(f"\n CLIP Fusion Model:")
 print(f" Test Accuracy: {fusion_result['test_accuracy']:.4f} ({fusion_result['test_accuracy']*100:.2f}%)")
 best_epoch_val = fusion_result.get('best_val_epoch', fusion_result.get('best_epoch', 'N/A'))
 print(f" Best Epoch: {best_epoch_val}/{fusion_result['total_epochs']}")
 
 # Performance comparison
 diff = fusion_result['test_accuracy'] - best_result['test_accuracy']
 diff_pct = diff * 100
 
 print(f"\n Fusion vs Best Single-Modal:")
 print(f" Difference: {diff:+.4f} ({diff_pct:+.2f} percentage points)")
 
 if diff > 0:
 print(f" Fusion model outperforms by {diff_pct:.2f}%!")
 elif diff < 0:
 print(f" Fusion model underperforms by {abs(diff_pct):.2f}%")
 else:
 print(f" Performance is equal")
 
 # Final recommendation
 print(f"\n Recommendation:")
 if fusion_result and fusion_result['test_accuracy'] > best_result['test_accuracy']:
 print(f" Use CLIP Fusion Model for best performance")
 else:
 print(f" Use {best_name} for best performance")
 
 print("\n" + "="*70)
 print(f" Evaluation Complete!")
 print("="*70)
 
 # Save report to JSON
 from datetime import datetime
 timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
 report_file = f"results/evaluation_report_{timestamp}.json"
 
 report_data = {
 'timestamp': timestamp,
 'single_modal_results': {k: {'test_accuracy': v['test_accuracy']} 
 for k, v in results_dict.items()},
 'best_single_modal': {
 'name': best_name,
 'accuracy': best_result['test_accuracy']
 }
 }
 
 if fusion_result:
 report_data['fusion_model'] = fusion_result
 report_data['comparison'] = {
 'difference': float(diff),
 'difference_percentage': float(diff_pct)
 }
 
 os.makedirs('results', exist_ok=True)
 with open(report_file, 'w') as f:
 json.dump(report_data, f, indent=4)
 
 print(f"\n Report saved: {report_file}")
 
 return report_data

# Generate report
if single_modal_results:
 report = generate_evaluation_report(single_modal_results, fusion_model_result)
else:
 print(" No results to generate report")

## Evaluation Complete!
